# optimizer-init-params-list — faded example 2: Materialize params list inside each param group

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-init-params-list`. The last cell reports your progress on the `PyTorch: Optimizer init` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Optimizer init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-init-params-list`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-init-params-list"
DD_SUBTOPIC = "PyTorch: Optimizer init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When an optimizer accepts param groups (a list of dicts), each group's `'params'` entry must be individually materialized into a list. The same generator-exhaustion problem applies per-group: once iterated in `.step()`, a generator-valued `'params'` is exhausted and subsequent steps update nothing.

## Faded exercise 2

Complete `GroupSGD.__init__`. The loop skeleton is provided. Inside the loop, build a new dict with the group's `'lr'` and its `'params'` MATERIALIZED into a list, then append it to `self.param_groups`.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

class GroupSGD:
    def __init__(self, param_groups):
        self.param_groups = []
        for g in param_groups:
            self.param_groups.append({'params': list(g['params']), 'lr': g['lr']})

    @t.no_grad()
    def step(self):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is not None:
                    p.data -= g['lr'] * p.grad

    def zero_grad(self):
        for g in self.param_groups:
            for p in g['params']:
                p.grad = None

# --- run it ---
t.manual_seed(0)
enc = nn.Linear(4, 4)
head = nn.Linear(4, 2)
opt = GroupSGD([{'params': enc.parameters(), 'lr': 0.001},
                {'params': head.parameters(), 'lr': 0.01}])
print([type(g['params']).__name__ for g in opt.param_groups])  # ['list', 'list']


def _test():
    import torch as t
    import torch.nn as nn
    t.manual_seed(0)
    enc = nn.Linear(4, 4)
    head = nn.Linear(4, 2)
    opt = GroupSGD([
        {'params': enc.parameters(),  'lr': 0.001},
        {'params': head.parameters(), 'lr': 0.01},
    ])
    assert len(opt.param_groups) == 2
    for g in opt.param_groups:
        assert isinstance(g['params'], list), 'each group params must be a list'
    assert opt.param_groups[0]['lr'] == 0.001
    assert opt.param_groups[1]['lr'] == 0.01
    assert len(opt.param_groups[0]['params']) == 2  # weight + bias
    assert len(opt.param_groups[1]['params']) == 2
    # Run 2 steps and confirm params updated
    prev_enc = opt.param_groups[0]['params'][0].data.clone()
    for _ in range(2):
        for g in opt.param_groups:
            for p in g['params']:
                p.grad = t.ones_like(p)
        opt.step()
    curr_enc = opt.param_groups[0]['params'][0].data
    assert not t.allclose(prev_enc, curr_enc)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

class GroupSGD:
    def __init__(self, param_groups):
        self.param_groups = []
        for g in param_groups:
            self.param_groups.append({'params': list(g['params']), 'lr': g['lr']})

    @t.no_grad()
    def step(self):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is not None:
                    p.data -= g['lr'] * p.grad

    def zero_grad(self):
        for g in self.param_groups:
            for p in g['params']:
                p.grad = None

# --- run it ---
t.manual_seed(0)
enc = nn.Linear(4, 4)
head = nn.Linear(4, 2)
opt = GroupSGD([{'params': enc.parameters(), 'lr': 0.001},
                {'params': head.parameters(), 'lr': 0.01}])
print([type(g['params']).__name__ for g in opt.param_groups])  # ['list', 'list']
```
</details>